# 1 Imports and Dataset

In [ ]:
!pip install -Uqq ipdb

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn import linear_model
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
import datetime as dt
from torch import nn
from torch.optim import Adam
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
import seaborn
from typing import Tuple
import torch
import ipdb
import sklearn.tree as tree
from sklearn  import metrics
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import RocCurveDisplay
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


##1.2 Load Dataset

In [ ]:
df = pd.read_excel('movies.xlsx') #load datatset
df

In [ ]:
pd.set_option('display.max_columns',None) #see all the columns

# 2.Data Cleaning

### 2.0.See data type for each column

In [ ]:
df.info()

In [ ]:
df.columns


We observe teh features. We replace 2 spaces with 1 space and then the space with '_'. Then we apply lower() function in order to convert features to lower_case.

In [ ]:
#fixed type of columns
new_columns=[col.strip().replace('  ',' ').replace(' ','_').lower() for col in df.columns]
df.columns=new_columns

In [ ]:
df.columns #check if done


### 2.1.Remove columns

We see where do we have missing values per column.We see that``` 'Distributor','IMDb Rating', 'IMDB vs RT disparity' ```columns have all their values null so we drop them.```Primary Genre``` column has 1382/1402 lines with missing values containing not important information,so again we drop it. ``` 'Film' ```column gives the title of the moovie, which is not again an important information for the classification so we also drop it.

In [ ]:
df.isna().sum() #σsee per column how many missing values we have

In [ ]:
df.drop(['film','primary_genre','distributor','imdb_rating','imdb_vs_rt_disparity'], axis=1, inplace=True) #drop columns

### 2.2.Handling missing values

In [ ]:
df.isnull().sum() #see where do we have null values /column

In [ ]:
# Calculating the percentage of missing values for each column
missing_data = df.isnull().sum()
missing_percentage = (missing_data[missing_data > 0] / df.shape[0]) * 100

# Prepare values
missing_percentage.sort_values(ascending=True, inplace=True)

# Plot the barh chart
fig, ax = plt.subplots(figsize=(15, 4))
ax.barh(missing_percentage.index, missing_percentage, color='#ff6200')

# Annotate the values and indexes
for i, (value, name) in enumerate(zip(missing_percentage, missing_percentage.index)):
    ax.text(value+0.5, i, f"{value:.2f}%", ha='left', va='center', fontweight='bold', fontsize=18, color='black')

# Set x-axis limit
ax.set_xlim([0, 40])

# Add title and xlabel
plt.title("Percentage of Missing Values", fontweight='bold', fontsize=22)
plt.xlabel('Percentages (%)', fontsize=16)
plt.show()

We see that columns ``` 'rotten_tomatoes_critics,'rotten_tomatoes_audience','genre'``` have 1 mising value, so we drop this line from each column

In [ ]:
df[df['oscar_detail'].isnull() | df['oscar_winners'].isnull()].head() #see lines  with null values

In [ ]:
df = df.dropna(subset=['rotten_tomatoes_critics','rotten_tomatoes_audience','genre'])
df=df.reset_index(drop=True)

In [ ]:
df.isna().sum() #check if done

### 2.3.Handling duplicates

In [ ]:
# Finding duplicate rows (keeping all instances)
duplicate_rows = df[df.duplicated(keep=False)]
print(duplicate_rows)

### 2.4.Handling with special characters like '-'

In the beggininhg we try to replace NA values of the columns ```'rotten_tomatoes_critics',rotten_tomatoes_audience'``` with the mean. We see that is not possible because in the data we have cells with '-' character.


In [ ]:
df.head(20) #see the type of our cells fro the first 20 lines


Search where is the '-' character

In [ ]:
df.tail(50)

In [ ]:
def replace_str_with_nan(x): #function to handle '-' character
    if not np.isreal(x): #check if x is not a real value
        if x == '-': #check if is character '-'
            return np.nan #replace with nan
    return x #else return the value

In [ ]:
df.isna().sum() #check again for missisng values

In [ ]:
for col in df.columns:
    df[col]=df[col].map(replace_str_with_nan) #apply teh function to every column of the daataframe

In [ ]:
df[~df['rotten_tomatoes_critics'].apply(np.isreal) |~df['metacritic_critics'].apply(np.isreal) ] #check if done

In [ ]:
df.isnull().sum() #check again for na values.Here we want to have na values because we substitute '-' with na values


In [ ]:
df[~df['rotten_tomatoes_critics'].apply(lambda x: isinstance(x, (int, float)))] #check if done

We have to transform columns (if needed) from object to int or float.

In [ ]:
#replace dtypes of columns where is needed
columns_to_convert=[
    'rotten_tomatoes_critics',
    'metacritic_critics',
    'average_critics',
    'metacritic_audience',
    'rotten_tomatoes_vs_metacritic_deviance',
    'average_audience',
    'audience_vs_critics_deviance'
]


In [ ]:
df[columns_to_convert]=df[columns_to_convert].astype(float) #convert to float

In [ ]:
df.dtypes #see dtypes of columns

Replace NA values in those columns with the mean of every column

In [ ]:
for col in columns_to_convert:
  mean_col=df[col].mean() #find mean for every column
  df[col].fillna(mean_col, inplace=True) #fill na with mean

In [ ]:
df.isna().sum() #check now if we have na values (we should not)

In [ ]:
df.tail(30) #check if done

In [ ]:
df.shape #check shape of dataframe

###2.5 Handling with columns that has data with symbol %

In [ ]:
#make regex to extract ('%') of every column that has this symbol in it's values and keep only the real number without the symbol
df['of_gross_earned_abroad']=df['of_gross_earned_abroad'].str.extract(r'(\d+\.\d+)')
df['budget_recovered_opening_weekend']=df['budget_recovered_opening_weekend'].str.extract(r'(\d+\.\d+)')
df['budget_recovered']=df['budget_recovered_opening_weekend'].str.extract(r'(\d+\.\d+)')

In [ ]:
df.head(3) #check if done

Convert the concrete columns to float

In [ ]:
#defien columsn  to convert
columns_to_convert=['of_gross_earned_abroad',
         'budget_recovered_opening_weekend',
         'budget_recovered']

In [ ]:
df[columns_to_convert] = df[columns_to_convert].astype(float)

In [ ]:
#check type of every column
df.dtypes

In [ ]:
df.tail(30) #check if the conversion to float  done

##2.6 Handling with columns opening_weekend,domestic_gross,foreign_gross,worldwide_gross,budget_($million)

We observe that the columns with ($ million) are simply a rounding of the corresponding columns, so we drop them.

In [ ]:
df.drop(['opening_weekend_($million)','domestic_gross_($million)','foreign_gross_($million)','worldwide_gross_($million)'],axis=1,inplace=True)

In [ ]:
df.columns #check if done

We check if the columns we kept have  symbols that prevent the conversion from object to float.

In [ ]:
df[~df['opening_weekend'].map(np.isreal)]

In [ ]:
df[~df['domestic_gross'].map(np.isreal)]

In [ ]:
df[~df['foreign_gross'].map(np.isreal)]

In [ ]:
df[~df['budget_($million)'].map(np.isreal)]

In [ ]:
df[~df['worldwide_gross'].map(np.isreal)]

We see that we have numbers with commas. So we replace commas with ''

In [ ]:
def replace_comma(x): #define function
  if np.isreal(x): #if the value is number
    return x #return x
  return x.replace(',','') #else replace ',' with nothing

In [ ]:
df['opening_weekend']=df['opening_weekend'].map(replace_comma)
df['domestic_gross']=df['domestic_gross'].map(replace_comma)
df['foreign_gross']=df['foreign_gross'].map(replace_comma)
df['worldwide_gross']=df['worldwide_gross'].map(replace_comma)


In [ ]:
df[~df['opening_weekend'].map(np.isreal)] #check if done

Convert columns

In [ ]:
columns_to_convert=['opening_weekend',
                    'domestic_gross',
                    'foreign_gross',
                    'worldwide_gross',
                    'budget_($million)']

In [ ]:
df[columns_to_convert] = df[columns_to_convert].astype(float)

In [ ]:
df.dtypes #check if done

## 2.7 Handling with column release_date_(us)

We observe that the format for date is not the same. We have values like
Nov 2,2007 and  November 16,2007

In [ ]:
df['release_date_(us)'].head(10)

In [ ]:
#convert to date type
df['release_date_(us)']=pd.to_datetime(df['release_date_(us)'])
df['release_date_(us)']=pd.to_datetime(df['release_date_(us)'],format="%m/%d/%Y") #convert to 'YY-MM-DD' format

In [ ]:
df['release_date_(us)'].head(10) #check if done

In [ ]:
#keep only year because is preferable for training
df['release_date_(us)']=df['release_date_(us)'].dt.year

In [ ]:
#convert column to int
df['release_date_(us)']=df['release_date_(us)'].astype(int)

In [ ]:
#find the most frequent year
df['release_date_(us)'].mode()

## 2.8 Handling with column oscar_winners and oscar_details

In [ ]:
df[(~df['oscar_winners'].isnull()) | (~df['oscar_detail'].isnull())] #check where have values at 'oscar_winners' or 'oscar_detail' columns

In [ ]:
df['oscar_winners']=df['oscar_winners'].str.lower() #lower case for handling the column oscar_winners better

In [ ]:
df[(~df['oscar_winners'].isnull()) | (~df['oscar_detail'].isnull())] #check if done

We observe that there are cases where, for example, the oscar detail is 'Best Actress,' while the entry does not have an oscar_winner but null, which is incompatible. Therefore, we replace null with oscar_winners.

In [ ]:
condition = (~df['oscar_detail'].isnull()) & (df['oscar_winners'].isnull())
df.loc[condition, 'oscar_winners'] = 'oscar winner'

In [ ]:
#df[(df['oscar_detail'] == 'Best Actress') &  (df['oscar_winners'].isnull())] #check if done
print("\nRows matching the condition after changes:")
df[condition]

Column oscar_detail has not any intrested information that will help on training so we drop it

In [ ]:
df.drop(['oscar_detail'], axis=1, inplace=True)

In [ ]:
def fillna_with_no(x): #define a function that map null values to 0 and not-null to 1
  if pd.isnull(x):
    return 0
  else:
    return 1

In [ ]:

df['oscar_winners'] = df['oscar_winners'].apply(fillna_with_no) #apply function in oscar_winners

In [ ]:
df['oscar_winners'] #check if done

In [ ]:
df[(df['oscar_winners'].isnull())] #check again for na values.There we expect not to have null values

In [ ]:
#convert int to float
df['oscar_winners']=df['oscar_winners'].astype(float)

##2.9 Columns budget_($million), budget_recovered,budget_recovered_opening_weekend

Search for NA values

In [ ]:
df[df['budget_($million)'].isnull()] #see which lines in column budget_($million) are null

Find mean of its column and replace NA values with mean

In [ ]:
mean_col=df['budget_($million)'].mean() #find mean
df['budget_($million)'].fillna(mean_col, inplace=True) #fill na with mean

In [ ]:
df[df['budget_($million)'].isnull()] #check if done

In [ ]:
df[df['budget_recovered'].isnull()] #the same goes with 'budget_recovered' column

In [ ]:
mean_col=df['budget_recovered'].mean() #find mean
df['budget_recovered'].fillna(mean_col, inplace=True) #fill na with mean
df[df['budget_recovered'].isnull()] #check if done

In [ ]:
mean_col=df['budget_recovered_opening_weekend'].mean() #find mean
df['budget_recovered_opening_weekend'].fillna(mean_col, inplace=True) #fill na with mean
df[df['budget_recovered_opening_weekend'].isnull()] #check if done

##2.10 Convertion of categorical variables to numerical

Multi-hot encoding for 'genre' column

####2.10.1 Column genre

In [ ]:
df.dtypes

In [ ]:
df.script_type.unique() #keep only the unique values of column 'script_type'

In [ ]:
df.genre.unique()#keep only the unique values of column 'genre'

In [ ]:
df['genre']=df['genre'].str.strip().str.lower() #with strip function ,erase tthe ' ' before or after the quotes and then convert to lower_case
df['genre'].unique() #check if done

In [ ]:
df['genre'].mode() #find the most frequent

We see that we have 'horro' and 'horror'.We assume that is the same,so we replace 'horro' with 'horror'

In [ ]:
#see where 'horro' is containing along with other features
df[df['genre'].str.contains(r'\bhorro\b')]

In [ ]:
df['genre']=df['genre'].str.replace(r'\bhorro\b', 'horror', regex=True)

In [ ]:
df[df['genre'].str.contains(r'adventure,\s*$')]#see where 'adventure,' is the last genre among other features

We replace 'adventure,' with 'adventure'

In [ ]:
df['genre']=df['genre'].str.replace(r'adventure,\s*$', 'adventure', regex=True)

In [ ]:
columns_of_genre=df['genre'].str.split(', ').explode().unique() #check if done and check also the unique genre type
df1=pd.DataFrame(columns=columns_of_genre)
df1
columns_of_genre

We replace 'sport'-'sports', 'famiily'-'family','thriler'-'thriller', 'music'-'musical', 'remake /reboot' - 'remake, reboot'

In [ ]:
df[df['genre'].str.contains(r'\bsport\b')] #see where 'sport' is containing along with other features

In [ ]:
df['genre']=df['genre'].str.replace(r'\bsport\b', 'sports', regex=True)#replace

In [ ]:
df[df['genre'].str.contains(r'\bfamiily\b')] #see where 'famiily' is containing along with other features
df['genre']=df['genre'].str.replace(r'\bfamiily\b','family') #replace with 'family'

In [ ]:
df[df['genre'].str.contains(r'\bthriler\b')] #see where 'thriler' is containing along with other features
df['genre']=df['genre'].str.replace(r'\bthriler\b','thriller') #replace with 'thriller'

In [ ]:
df[df['genre'].str.contains(r'\bmusic\b')] #see where 'music' is containing along with other features
df['genre']=df['genre'].str.replace(r'\bmusic\b','musical') #replace with 'musical'

We replace 'sci-fi. thriller' με 'sci-fi,thriller'

In [ ]:
df[df['genre'].str.contains(r'sci-fi\. thriller')] #see where ''sci-fi\. thriller' is containing

In [ ]:
df['genre']=df['genre'].str.replace(r'sci-fi\. thriller','sci-fi, thriller')

We use evey unique category of column 'genre' and make a dataframe with column names of the unique categories.

In [ ]:
columns_of_genre=df['genre'].str.split(', ').explode().unique() #split genre and get the unique values from everry line in column genre
df1=pd.DataFrame(columns=columns_of_genre) #make a dataframe with columns_of_genre
#df1
columns_of_genre #see all the uniqye values

In [ ]:
df.dtypes

In [ ]:
df=pd.concat([df,df1]) #concat two dataframes

In [ ]:
df['year']=df['year'].astype(int) #μου αλλαζει το year οταν κανω το concat οπότε το μετατρέπω ξανά σε int

In [ ]:
df.dtypes

In [ ]:
for i,row in enumerate (df['genre']): #loop all the lines of column 'genre'
  l=(row.split(', ')) #split the line to get the different types of 'genre'
  for item in l:
    #print(item)
    df.loc[i,item]=1 #put 1 in the df where the type of genre is appeared

In [ ]:
df[columns_of_genre]=df[columns_of_genre].fillna(0) #put 0 to all the other lines

In [ ]:
df #check if done

In [ ]:
#drop column genre
df.drop(['genre'], axis=1, inplace=True)

####2.10.2 Column script_type


We want to do multi-hot encoding for column  'script_type'.

In [ ]:
df['script_type']=df['script_type'].str.strip().str.lower() #erase ' ' before or after quotes  and then to lower_case
df['script_type'].unique() #see unique values

In [ ]:
df['script_type'].mode() #find the most common script_type

In [ ]:
columns_of_script_type=df['script_type'].str.split(', ').explode().unique() #split by comma and get the unique values of column 'script_type'
columns_of_script_type

In [ ]:
df2=pd.DataFrame(columns=columns_of_script_type) #make a dataframe with the unique values of column 'script_type'
df2

In [ ]:
df=pd.concat([df,df2]) #concat the 2 datafames
df

In [ ]:
for i,row in enumerate (df['script_type']):
  l=(row.split(', ')) #split every line by comma
  for item in l: #where the item is appeared put 1
    #print(item)
    df.loc[i,item]=1

In [ ]:
df[columns_of_script_type]=df[columns_of_script_type].fillna(0) #put to every other column 0

In [ ]:
df.dtypes #check type of every column

In [ ]:
df['year']=df['year'].astype(int)

In [ ]:
df

In [ ]:
#drop column script type
df.drop(['script_type'], axis=1, inplace=True)

In [ ]:
col=['documentary','biography','remake','original screenplay','sequel','based on a true story','remake / reboot','prequel','adaptation']
df[col] = df[col].astype(float) #convert to float type

###2.11 Info about the final dataframe

In [ ]:
df.dtypes

In [ ]:
df.shape
df

In [ ]:
df.isna().sum()

In [ ]:
#correlation between the features
import seaborn as sns
sns.heatmap(df.corr())



#3.Define Model and Train

In this section we apply diffetrent classification algorithms with diffrent approaches(due to imbalanced data). We evaluate every approach and try to figure out which gives the best performance.

##3.1Train_set and Test_set

In [ ]:
X= df.drop(['oscar_winners'], axis=1)
y=df['oscar_winners']
X.shape, y.shape

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)
X_train.shape, X_test.shape, y_train.shape, y_test.shape
#((980, 47), (420, 47), (980,), (420,))

In [ ]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

##3.2 Standardization

Here we perform the standardization technique in order to transform the data before applying any machine algorithm

In [ ]:
#Standardization
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

scaler.fit(X_train) #let scaler to fit to X_train #find the mean and std of the data
X_train=scaler.transform(X_train) #let scaler to transform the data
X_test=scaler.transform(X_test) #transform also for unseeen data(with the same parameters
                                #learned from .fit)


In [ ]:
type(y_train), type(X_train)

##3.3 Useful functions

Function ```flatten_dict``` flatten our metrics from the classification_report in order to make a dataframe with the total results

In [ ]:
def flatten_dict(report_dict):
  result_dict={} #initialize a dict
  for k1, k2 in report_dict.items():
    if k1=='accuracy' or k1=='name':
      result_dict[k1]=k2
    else:
      for col, v in k2.items():
          result_dict[k1+'_'+col]=v
  return result_dict



Function ```confusionMatrix_and_roc``` is a function that plots the roc curve and the confusion matrix. We will use this function in order to evaluate the performance of every model

In [ ]:
def confusionMatrix_and_roc(classifier,X_test,y_test):

  roc_display=RocCurveDisplay.from_estimator(
      classifier,
      X_test,
      y_test)


  confusion_matrix=ConfusionMatrixDisplay.from_estimator(
      classifier,
      X_test,
      y_test

  )


  return fig


In [ ]:

type(X_train),type(y_train),type(X_test), type(y_test)

##3.4 Attempt with Logistic Regression model

###3.4.1 Logistic Regression Model

In [ ]:
lr1=LogisticRegression()
lr1.fit(X_train,y_train)
y_pred=lr1.predict(X_test)
print("Classification report:\n",metrics.classification_report(y_test,y_pred))
lr_class_report=metrics.classification_report(y_test,y_pred, output_dict=True)
lr_class_report['name']='Logistic Regression'
plt.show(confusionMatrix_and_roc(lr1,X_test, y_test))

From roc curve we can infer that the model performance is good, because the curve has almost the ideal behaviour(to be on the left-top corner).We see that the model can predict very well the class 0.0 but not the class 1.0 beacuse of the number of non-oscar moovies in the dataset. Also we see that we have a very high AUC=0.87 which means that our model can distinguish  well between the 2 classes.The macro-avg recall and presicion is high and the accuracy value is very high,indicating that the model can predict well out of all instances.

###3.4.2 Balanced Logistic Regression Model

We try to balance the classes through our Logistic Regression model and evaluate it again

Trying Logistic regression classifier with balanced classes we see that we get a higher score on recall for class 1. This behavior occurs because we set weights to each class according with the frequency of its class. So we try to balance the amount of non-oscar and oscar moovies and fit the model again. Also we see again a good roc diagram with AUC=0.88,which means that the model can predict the actual positive values with minor mistakes. The only disadvantage for this method is that we get a low presicion for class 1.0  which means that the model still struggles to predict which moovies got oscar. This is propably because of the imbalance of the data.

In [ ]:
lr2=LogisticRegression(class_weight="balanced")
lr2.fit(X_train,y_train)
y_pred=lr2.predict(X_test)
print("Classification report:\n",metrics.classification_report(y_test,y_pred))
lr_class_report_balanced=metrics.classification_report(y_test,y_pred, output_dict=True)
lr_class_report_balanced['name']='Logistic Regression Balanced'
plt.show(confusionMatrix_and_roc(lr2,X_test,y_test))

###3.4.3 Logistic Regression Model with fixed weights

We try now with fixed class weights

Applying fixed weights on classes we see that we get better results for class 1.0 than with balanced classes. We get also a higher macro avg score. From the roc curve we can infer that the model performance is good because the curve **tends to** be on the top-left corner meaning that every instance that the model identifies as belonging to the positive class actually belongs to that class.The auc score is high. So, in generall we could say that applying fixed weights on classes using logistic regression model,model's performance is good, but not perfect.

In [ ]:
lr3=LogisticRegression(class_weight={0:0.1 , 1:0.9})
lr3.fit(X_train,y_train)
y_pred=lr3.predict(X_test)
print("Classification report:\n",metrics.classification_report(y_test,y_pred))
lr_class_report_custom_weights_1=metrics.classification_report(y_test,y_pred, output_dict=True)
lr_class_report_custom_weights_1['name']='Logistic Regression with custom weights 1'
plt.show(confusionMatrix_and_roc(lr3,X_test,y_test))

Same as before we apply a weight of 0.05 in class 0 and a weight of 0.95 on class 1 due to the imbalanced data. This approach has no difference comparing to the previous, except that here the presicion score for class 1 is lower.

In [ ]:
lr4=LogisticRegression(class_weight={0:0.05 , 1:0.95})
lr4.fit(X_train,y_train)
y_pred=lr4.predict(X_test)
print("Classification report:\n",metrics.classification_report(y_test,y_pred))
lr_class_report_custom_weights_2=metrics.classification_report(y_test,y_pred, output_dict=True)
lr_class_report_custom_weights_2['name']='Logistic Regression with custom weights 2'
plt.show(confusionMatrix_and_roc(lr4,X_test,y_test))

##3.5 Attempt with Desicion Tree Classifier

###3.5.1 Simple Desicion Tree Classifier

Using decision tree classifier we get a high presicion and recall for class 1.0 comparing to the above approaches.

In [ ]:
dt1=tree.DecisionTreeClassifier() #trying to get hiqh recall fro class oscar_winners
dt1.fit(X_train,y_train) #fit
y_pred=dt1.predict(X_test) #predict
print("Classification report:\n",metrics.classification_report(y_test,y_pred))
dt_class_report=metrics.classification_report(y_test,y_pred, output_dict=True)
#plt.show(confusionMatrix_and_roc(dt1,X_test,y_test))
dt_class_report['name']='DT clf'

###3.5.2 Balanced Desicion Tree Classifier

Using Desision Tree Classifier with balanced class weights, we have again a very good prediction for class 0.0. The model can't predict very well class 1.0 due to imbalanced data.Comparing to the previous mdel the results for class 1.0 are worse.

In [ ]:
dt2=tree.DecisionTreeClassifier( class_weight="balanced") #trying to get hiqher recall fro class oscar_winners
dt2.fit(X_train,y_train) #fit
y_pred=dt2.predict(X_test) #predict
print("Classification report:\n",metrics.classification_report(y_test,y_pred))
dt_class_balanced_report=metrics.classification_report(y_test,y_pred, output_dict=True)
dt_class_balanced_report['name']='DT balanced'
#dt_class_balanced_report

###3.5.3 Weighted_class Desicion Tree Classifier

Desicion Tree classifier with fixed class weights can predict very weel the instances on class 0.0 but still struggling to make correct prediction for class 1.0. Comparing to the previous approach we get worse results for class 1.0 and also for macro avg scores.

In [ ]:
dt3=tree.DecisionTreeClassifier( class_weight={0:0.1, 1:0.9}) #trying to get hiqher recall and presiocn for class oscar_winners
dt3.fit(X_train,y_train) #fit
y_pred=dt3.predict(X_test) #predict
print("Classification report:\n",metrics.classification_report(y_test,y_pred))
dt_class_custom_weights1_report=metrics.classification_report(y_test,y_pred, output_dict=True)
dt_class_custom_weights1_report['name']='DT custom_weights1'


Applying smaller class weights on class 0, the model performs  somehow better result but with no significant difference.

In [ ]:
clf=tree.DecisionTreeClassifier( class_weight={0:0.05, 1:0.95}) #trying to get hiqher recall and presicion scores for class oscar_winners
clf=clf.fit(X_train,y_train) #fit
y_pred=clf.predict(X_test) #predict
print("Classification report:\n",metrics.classification_report(y_test,y_pred))
dt_class_custom_weights2_report=metrics.classification_report(y_test,y_pred, output_dict=True)
dt_class_custom_weights2_report['name']='DT custom_weights_2'


##3.6 Attempt with KNN

###3.6.1 Kneighbors Classifier

Using knn classifier we can see that our model can predict very well the class 0.0(from high presicion,recal and F1-score). However the performance for the class 1.0 is not as strong. Also from the roc diagram we can infer that after predicting the 60% of the actual posituve instances in the daataset ,the curve is very close to the diagonal line which means that the model predicts almost by chance.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier as KN
knn_clf=KN(weights='uniform') #trying to get hiqh recall fro class oscar_winners
knn_clf.fit(X_train,y_train) #fit
print("Classification report\n",metrics.classification_report(y_test,y_pred))
knn_class_report_uniform=metrics.classification_report(y_test,y_pred, output_dict=True)
knn_class_report_uniform['name']='KNN_uniform'
plt.show(confusionMatrix_and_roc(knn_clf,X_test,y_test))
#knn_class_report_uniform#

###3.6.2 KNeighborsClassifier with distance weights

With 'distance' weights knn algorithm still struggling to predict right for class 1.0.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier as KN
knn_clf=KN(weights='distance') #trying to get hiqh recall fro class oscar_winners
knn_clf.fit(X_train,y_train) #fit
print("Classification report\n",metrics.classification_report(y_test,y_pred))
knn_class_report_dist=metrics.classification_report(y_test,y_pred, output_dict=True)
knn_class_report_dist['name']='KNN_distance'
plt.show(confusionMatrix_and_roc(knn_clf,X_test,y_test))


##3.7 Attempt with naive classifier

###3.7.1 Gausian NB Classifier

For class 0, the model has a high precision , indicating that when it predicts an instance as class 0, it's usually correct. However, for class 1, the precision is low , suggesting that when the model predicts an instance as class 1, it's often incorrect. The model has a high recall for class 1, indicating that it effectively captures most of the actual class 1 instances. However, for class 0, the recall is low, meaning it misses a significant portion of the actual class 0 instances.Also the F1-score for class 0 is 40% ,while for class 1 is quite low, indicating a tarde-off between presicion and recall for class 0 and an imbalanced tilted toward recall for class 1.The overall accuracy is very low suggesting that the model's predictions are not very accurate for both classes.Also the macro avg F1-score is very low. Overall, the model is performing bad, especially in correctly identifying class 0 instances, despite high precision in this class. ALso from the roc seems that the model predict by chance.

In [ ]:
from sklearn.naive_bayes import GaussianNB as GNB
gnb_clf=GNB() #trying to get hiqh recall fro class oscar_winners
gnb_clf.fit(X_train,y_train) #fit
y_pred=gnb_clf.predict(X_test)
#print(y_pred)
print("Classification report\n",metrics.classification_report(y_test,y_pred))
gnb_class_report=metrics.classification_report(y_test,y_pred, output_dict=True)
gnb_class_report['name']='GNB'
plt.show(confusionMatrix_and_roc(gnb_clf,X_test,y_test))


##3.8 Attempt with support vector machines

Now we apply ```SVM``` algorithm for classification

Despite the high presicion for class 0.0 we see that the model i sonly correct 6% when trying to redict class 1.0. Also the model finds correct all instances of class 1.0 but identifies only 24% of all actual class 0.0 instances.THe macro avg presicion , presiscion and F1-score are low. The model performs well in terms of precision for class 0.0 but poorly for class 1.0.
For class 0.0, the recall is low, indicating that the model misses many actual instances of this class.
Class 1.0 has perfect recall but very low precision, suggesting that while the model captures all instances of this class, it also misclassifies many other instances as belonging to this class. The model tries so much to find the instances of class 1.0, so it makes a lot of mistakes for class 0.0.The recall score of class 1.0 is perfect so it finds all the moovies with oscar.But the presicion for class 1.0 is very low ,so the model tries to predict well the instances of class 1.0,doing a lot of mistakes on class 1.0(FPR). Hoewever the model makes perfect predictions for class 0.0,(presicion of class 0.0).

####3.8.1 SVM with linear kernel

In [ ]:
from sklearn import svm
svm1=svm.SVC(kernel="linear")
svm1.fit(X_train,y_train)
#print(svm.predict(X_test))
print("CLassification report\n",metrics.classification_report(y_test,y_pred))
svm_class_report=metrics.classification_report(y_test,y_pred,output_dict=True)
svm_class_report['name']='SVM'
plt.show(confusionMatrix_and_roc(svm1,X_test,y_test))
#svm_class_report



###3.8.2 SVM with linear kernel and weighted classes

Trying svm algorothm with linear kernel and fixed class weights we see no difference in the performance of the model.Still the presicion for class 1.0 is as low as the recall for class 0.0.

In [ ]:
from sklearn.svm import SVC
svm2=SVC(kernel="linear",class_weight={0:0.30,1:0.70})
svm2.fit(X_train,y_train)
#print(svm.predict(X_test))
print("CLassification report\n",metrics.classification_report(y_test,y_pred))
svm_class_report_weights=metrics.classification_report(y_test,y_pred,output_dict=True)
svm_class_report_weights['name']='SVM weighted_class'
plt.show(confusionMatrix_and_roc(svm2,X_test,y_test))
#svm_class_report_weights


###3.8.3 SVM with 'poly' kernel and weighted classes

Using svm whith polz kernl and class weights we observe no difference in the performance of our model.

In [ ]:
svm3=SVC(kernel="poly",class_weight={0:0.30,1:0.70})
svm3.fit(X_train,y_train)
#print(svm.predict(X_test))
print("CLassification report\n",metrics.classification_report(y_test,y_pred))
svm_class_report_weights_poly=metrics.classification_report(y_test,y_pred,output_dict=True)
svm_class_report_weights_poly['name']='SVM weighted with polynomial kernel'
plt.show(confusionMatrix_and_roc(svm2,X_test,y_test))
#svm_class_report_weights

###3.8.4 SVM with 'rbf' kernel and weighted classes

Here we use rbf kernel and clas weights but still we get a poor perforamnce

In [ ]:
svm4=SVC(kernel="rbf",class_weight={0:0.30,1:0.70})
svm4.fit(X_train,y_train)
#print(svm.predict(X_test))
print("CLassification report\n",metrics.classification_report(y_test,y_pred))
svm_class_report_weights_rbf=metrics.classification_report(y_test,y_pred,output_dict=True)
svm_class_report_weights_rbf['name']='SVM weighted with rbf kernel'
plt.show(confusionMatrix_and_roc(svm3,X_test,y_test))
#svm_class_report_weights

###3.8.5 SVM with 'linear' kernel, C=0.1 and gamma='scale'

We try here to use svm algorithm with parameters C=0.1, a linear kernel and a gamma='scale'. aThe pwrformance remains the same.

In [ ]:
svm5=SVC(C=0.1,kernel="linear",gamma='scale')
svm5.fit(X_train,y_train)
#print(svm.predict(X_test))
print("CLassification report\n",metrics.classification_report(y_test,y_pred))
svm_class_report_weights_gamma_scale=metrics.classification_report(y_test,y_pred,output_dict=True)
svm_class_report_weights_gamma_scale['name']='SVM with C=0.1 and gamma=scale'
plt.show(confusionMatrix_and_roc(svm4,X_test,y_test))
#svm_class_report_weights

###3.8.6 SVM with 'linear' kernel C=0.1 and gamma='auto'

In [ ]:
svm6=SVC(C=0.1,kernel="linear",gamma='auto')
svm6.fit(X_train,y_train)
#print(svm.predict(X_test))
print("CLassification report\n",metrics.classification_report(y_test,y_pred))
svm_class_report_weights_gamma_auto=metrics.classification_report(y_test,y_pred,output_dict=True)
svm_class_report_weights_gamma_auto['name']='SVM C=0.1 and gamma auto'
plt.show(confusionMatrix_and_roc(svm5,X_test,y_test))
#svm_class_report_weights

##3.9 Collect all metrics and plot

We gather all our metrics from the classification algorithms to a dataframe(resulted_df),using function ```flatten_dict```

In [ ]:
                        #Logistic Regression Classifier
result_df=pd.DataFrame([flatten_dict(lr_class_report),
                        flatten_dict(lr_class_report_balanced),
                        flatten_dict(lr_class_report_custom_weights_1),
                        flatten_dict(lr_class_report_custom_weights_2),

                        #Desicion Tree Classifier
                        flatten_dict(dt_class_report),
                        flatten_dict(dt_class_balanced_report),
                        flatten_dict(dt_class_custom_weights1_report),
                        flatten_dict(dt_class_custom_weights2_report),

                        #KNN
                        flatten_dict(knn_class_report_uniform),
                        flatten_dict(knn_class_report_dist),

                        #GNB
                        flatten_dict(gnb_class_report),

                        #SVM
                        flatten_dict(svm_class_report),
                        flatten_dict(svm_class_report_weights),
                        flatten_dict(svm_class_report_weights_poly),
                        flatten_dict(svm_class_report_weights_rbf),
                        flatten_dict(svm_class_report_weights_gamma_scale),
                        flatten_dict(svm_class_report_weights_gamma_auto),

                        ])

name_column=result_df.pop('name')
result_df.insert(0,'name',name_column)
result_df

Then we plot from the dataframe only the presicion and recall for class 1.0 (as we try mostly to evaluate our model considering metrics for class 1.0 beacause it has the less instances fromclass 0,0), the macro_vg f1-score and accuracy,

In [ ]:
#plt.figure(figsize=(9000,58))
result_df.plot(x="name",y=['1.0_precision','1.0_recall','macro avg_f1-score','accuracy'], kind="bar")
plt.title('Classification Metrics Across Models')
plt.legend(loc='lower left',bbox_to_anchor=(1,0.5))
plt.show()

From the plot we could say that the Logistic Regression,Desicion Tree Classifier wth class weight 1,Logistic Regression with custom weight 1 and Logistic Regression with custom weight 2 have the best metrcis. Comparing with the other classifications algorithms they have a high 1.0_recall(indicating that the model can find the most instances of class 1.0)in combination with a high presicion.Also they have a high accuracy and good F1-scores. So we make a Voting classifier, which combines the 3 of them.

In [ ]:
#combine 3 best classifiers
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import cross_validate
clf1=LogisticRegression()
clf2=tree.DecisionTreeClassifier( class_weight={0:0.1, 1:0.9})
clf3=LogisticRegression(class_weight={0:0.1, 1:0.9})
eclf1=VotingClassifier(estimators=[('lr',clf1),('dt_w',clf2),('lr_w',clf3)],voting='hard')
eclf1.fit(X_train,y_train)
pred=eclf1.predict(X_test)
scores = cross_validate(clf,X_train,y_train, scoring=('accuracy','precision','recall'), cv=5)
print("Accuracy: %0.2f " % (scores['test_accuracy'].mean()))
print("Presicion: %0.2f " % (scores['test_precision'].mean()))
print("Recall: %0.2f " % (scores['test_recall'].mean()))
#print(scores['test_recall'])


Presicion for class 1.0 is low meaning that the voting classifier can only predict accurate a small set our data.Recall is also low meanining that the classifier can pedict a small number of the oscar-moovies.Accuracy is high,due to the high metrics for class 0.0. So we observe no difference in metrics with the classifier

## 3.10 KFold validation

KFodl_fnc is a function in order to do the 5-fold cross validation for every algorithm.

In [ ]:
def KFold_fnc(method_name):
  fold_results=[]
  from sklearn.model_selection import KFold as KFold
  kf =KFold(n_splits=5, shuffle=True)
  for i, (train_index, test_index) in enumerate (kf.split(X)):
    _X_train=X.values[train_index]
    _y_train=y.values[train_index]
    _X_test=X.values[test_index]
    _y_test=y.values[test_index]

    scaler = StandardScaler()
    scaler.fit(_X_train) #let scaler to fit to X_train #find the mean and std of the data
    _X_train=scaler.transform(_X_train) #let scaler to tranform the data
    _X_test=scaler.transform(_X_test) #transform also for unseeen data(with the same parameters
                                    #learned from .fit)
    clf=method_name
    clf=clf.fit(_X_train, _y_train) #fit
    #clf.get_depth()  #10
    y_pred=clf.predict(_X_test) #predict
    print('Fold', i, ':')
    print(metrics.classification_report(_y_test, y_pred))
    result_dict = metrics.classification_report(_y_test, y_pred, output_dict=True)
    result_dict['name'] = 'fold ' + str(i)+str(method_name)
    #plt.show(confusionMatrix_and_roc(clf,_X_test,_y_test))
    fold_results.append(flatten_dict(result_dict))
  return pd.DataFrame(fold_results)



In [ ]:
def plot_results(results,model):
  #plt.figure(figsize=(9000,58))
  results.plot(x="name",y=['1.0_precision','1.0_recall','macro avg_f1-score','accuracy'], kind="bar")
  plt.title('Metrics across Folds'+' '+'for'+' '+(model))
  plt.legend(loc='lower left',bbox_to_anchor=(1,0.5))
  plt.show()

In [ ]:
def mean_of_folds(results,method_name):
  presicion_mean=results['1.0_precision'].mean()
  recall_mean=results['1.0_recall'].mean()
  macro_mean=results['macro avg_f1-score'].mean()
  accuracy_mean=results['accuracy'].mean()
  df_mean = pd.DataFrame({
    '1.0_precision_mean': [presicion_mean],
    '1.0_recall_mean': [recall_mean],
    'macro avg_f1-score_mean': [macro_mean],
    'accuracy_mean': [accuracy_mean]
  })
  df_mean["name"]=method_name
  return df_mean


In [ ]:
def plot_mean_of_folds(df_mean):
  return df_mean.plot(x='name',y=['1.0_precision_mean','1.0_recall_mean','macro avg_f1-score_mean','accuracy_mean'], kind="bar")


In [ ]:
results1=(KFold_fnc(LogisticRegression()))

For class 0.0 , precision ranges between 0.96 to 1.0 indicating that when the model predicts class 0.0 it's accurate in its predictions. For class 1.0 oresicion ratings from 0.18 to 0.28, suggetsing that the mpodel struggling with the precise predictions for class 1.0. Class 0.0's recall ranges between 0.87 to 0.92, showing that the model identifies most of the actual class 0.0 instances. Class 1.0's recall varies from 0.47 to 0.89 indicating differing abilities to capture class 1.0 instances across folds.F1-scores are high reflecting a good balance between presiscion and recall for class 0.0.For class 1.0 F1-scores are lower due to lower presicion and recall for this class. Also accuracy ranges from 0.87 to 0.91 showing the overall correctness of the model;s predictions.However, the accuracy mightbe influenced from the instances of the majority class.So, the model predicts well in predicting the majority class with high presicion,recalland F1-scores. For class 0.0 the model's performance is worse,where precision,recall and F1-scores are lower across folds.

In [ ]:
plt1=plot_results(results1,'LogisticRegression')


In [ ]:
mean_df1=mean_of_folds(results1,'Logistic Regression')
mean1=plot_mean_of_folds(mean_df1)

In [ ]:
results2=(KFold_fnc(LogisticRegression(class_weight="balanced")))

In [ ]:
plt2=plot_results(results2,'Logistic Regression with balanced class weights')

Presicion for class 0.0 ranges from 0.98-1.00, indicating that when the model predicts some instances of class 0.0 is almost correct.The recall for this class ranges form 0.87-0.91 ,showing that the model can predict 0.0 class very well. Presicion for class 1.0 ranges  from 0.17 -0.25 ,suggestting that the model struggles with prescise predictions for class 1.0. Also 1.0 recall varies from 0.64-0.90, indicating different abillities to capture class 1.0 across folds.  F1-scores fro class 0.0 are high because of the high recall and precesion.For class 1.0 is lower due to low recall and precesion across folds.Accuracy ranges from 0.88-0.91 showing the overall correctness of the model's predictions.Howevere it may influenced by the number of instances for class 0.0.In generall the model can predict very well class 0.0 but for class 1.0 can't make such good predictions.

In [ ]:
mean_df2=mean_of_folds(results2,'Logistic Regression with balanced')
mean2=plot_mean_of_folds(mean_df2)

In [ ]:
results3=(KFold_fnc(LogisticRegression(class_weight={0:0.1 , 1:0.9})))

In [ ]:
plot_results(results3,'LR with fixed weights(0:0.1 1:0.9)')

For class 0.0 , precision ranges between 0.98 to 1.0 indicating that when the model predicts class 0.0 it's accurate in its predictions. For class 1.0 presicion ratings from 0.17 to 0.33, suggetsing that the model can not predict correct for class 1.0. Class 0.0's recall ranges between 0.91 to 0.95, showing that the model identifies most of the actual class 0.0 instances. Class 1.0's recall varies from 0.60 to 0.89 indicating differing abilities to capture class 1.0 instances across folds.F1-scores are high reflecting a good balance between presiscion and recall for class 0.0.For class 1.0 F1-scores are lower due to lower presicion and recall for this class. Also accuracy is very high  showing the overall correctness of the model's predictions.However, the accuracy might be influenced from the instances of the majority class.So, the model predicts well in predicting the majority class with high presicion,recal land F1-scores. For class 0.0 the model's performance is worse,where precision,recall and F1-scores are lower across folds.

In [ ]:
mean_df3=mean_of_folds(results3,'Logistic Regression 0:0.1, 1:0.9')
mean3=plot_mean_of_folds(mean_df3)

In [ ]:
results4=(KFold_fnc(LogisticRegression(class_weight={0:0.05 , 1:0.95})))

In [ ]:
plot_results(results4,'LR with fixed weights {0:0.05, 1:0.95}')

For class 0.0 , precision ranges between 0.98 to 1.0 indicating that when the model predicts class 0.0 it's accurate in its predictions. For class 1.0 presicion ratings from 0.17 to 0.42, suggetsing that the model struggling with the precise predictions for class 1.0. Class 0.0's recall ranges between 0.84 to 0.95, showing that the model identifies most of the actual class 0.0 instances. Class 1.0's recall varies from 0.62 to 1.00 showing that the model can find sometimes perfect the instances of class 1.0.In this situation(fold 1) the model try so hard to predict the oscar-moovies, but does a lot of mistakes on identifying them right(precesion for class 1.0 is 0.17).F1-scores are high reflecting a good balance between presiscion and recall for class 0.0.For class 1.0 F1-scores are lower due to lower presicion and recall for this class. Also accuracy is very high showing the overall correctness of the model's predictions.However, the accuracy might be influenced from the instances of the majority class.So, the model predicts well in predicting the majority class with high presicion,recall and F1-scores. For class 0.0 the model's performance is worse,where precision,recall and F1-scores are lower across folds(due to imbalanced data)

In [ ]:
mean_df4=mean_of_folds(results4,'Logistic Regression 0:0.05, 1:0.95')
mean4=plot_mean_of_folds(mean_df4)

In [ ]:
results5=(KFold_fnc(tree.DecisionTreeClassifier()))

In [ ]:
plot_results(results5,'Decision Tree classifier')

Presicion for class 0.0 ranges from 0.97-0.99, showing that the model when try to predict class 0.0 is very acccurate on its predictions.Recall for this class is also high, menaning that the model can predict weel the instances of class 0.0. In the opposite, presisicon for class 1.0 ranges from  0.11-0.50 and that shows that our model is struggling to preidct accurate the instances of this class. Also class 1.0's recall ranges from 0.42-0.46,indicating that the model's performance for class 1.0 is worse than predictions by chance.  Accuracy again is high due to the hιgh presicion and recall for class 0.0.

In [ ]:
mean_df5=mean_of_folds(results5,'DT')
mean5=plot_mean_of_folds(mean_df5)

In [ ]:
results6=KFold_fnc(tree.DecisionTreeClassifier( class_weight="balanced"))

In [ ]:
plot_results(results6,'DT balanced')

For class 0.0 , precision ranges between 0.95 to 0.99 indicating that when the model predicts class 0.0 it's accurate in its predictions. For class 1.0 presicion ratings from 0.23 to 0.62, suggetsing that the model can not predict correct for class 1.0. Class 0.0's recall ranges between 0.96 to 0.99, showing that the model identifies most of the actual class 0.0 instances. Class 1.0's recall varies from 0.23 to 0.62 indicating differing abilities to capture class 1.0 instances across folds.F1-scores are high reflecting a good balance between presiscion and recall for class 0.0.For class 1.0 F1-scores are lower due to lower presicion and recall for this class. Also accuracy is very high  showing the overall correctness of the model's predictions.However, the accuracy might be influenced from the instances of the majority class.So, the model predicts well in predicting the majority class with high presicion,recal land F1-scores. For class 0.0 the model's performance is worse,where precision,recall and F1-scores are lower across folds.

In [ ]:
mean_df6=mean_of_folds(results6,'DT Balanced')
mean6=plot_mean_of_folds(mean_df6)

In [ ]:
results7=KFold_fnc(tree.DecisionTreeClassifier( class_weight={0:0.1, 1:0.9}))

In [ ]:
plot_results(results7,'DT with fixed class weights {0:0.1, 1:0.9} ')

Precision,Recall ,F1-score for class 0.0:
  Presicion values from 0.95 to 0.99

indicating accurate predictions whebn the model identifies this class.
  Recall values are also high indicating that the model captures most actual instances of class 0.0
  F1-scores are also high.


Presicion,Recall F1-score for class 1.0:





In [ ]:
mean_df7=mean_of_folds(results7,'DT 0:0.1 1:0.9')
mean7=plot_mean_of_folds(mean_df7)

In [ ]:
results8=KFold_fnc(tree.DecisionTreeClassifier( class_weight={0:0.05, 1:0.95}))

In [ ]:
plot_results(results8,'DT with fixed class wights{0:0.05, 1:0.95}')

For class 0.0 , precision ranges between 0.98 to 0.96 indicating that when the model predicts class 0.0 it's accurate in its predictions. For class 1.0 presicion ratings from 0.27 to 0.57, suggetsing that the model can not predict correct for class 1.0. Class 0.0's recall is very high, showing that the model identifies most of the actual class 0.0 instances. Class 1.0's recall is low indicating differing abilities to capture class 1.0 instances across folds.F1-scores are high reflecting a good balance between presiscion and recall for class 0.0.For class 1.0 F1-scores are lower due to lower presicion and recall for this class. Also accuracy is very high,( because it might be influenced by majority class)  showing the overall correctness of the model's predictions. So ,again the model can make very good prediction for class 0.0 but is not so accuarte for class 1.0

In [ ]:
mean_df8=mean_of_folds(results8,'DT 0:0.05, 1:0.95')
mean8=plot_mean_of_folds(mean_df8)

In [ ]:
results9=KFold_fnc(KN(weights='distance'))

In [ ]:
plot_results(results9,'KNN weights=distance')

In [ ]:
plot_results(results9,'KNN weights=distance')

For class 0.0 presiscion and recall arae very high,indicating that when the model try to predict the non-oscar moovies is accuarate(0.94- 0.98) and when it tries to find  non-oscar moovies it identifies them correct(0.99-1.00). F1 -scores are also very high.
For class 1.0 precesion,recall and F1-scores are lower ,showing the poor abbillity of the model to identify and predict oscar-values.Accuracy is also high but driven by the accuarate predictiosn of the majority class.SO, in summary model performs consistenly well in predicting class 0.0 ,but it struggles with the minority class.

In [ ]:
mean_df9=mean_of_folds(results9,'KNN weight=distance')
mean9=plot_mean_of_folds(mean_df9)

In [ ]:
results10=KFold_fnc(GNB())

In [ ]:
plot_results(results10,'GNB')

Precision is high for Class 0.0, indicating a good ability to correctly identify this class when predicted. However, recall is significantly low, suggesting a high number of false negatives.This is because the model tries a lot to predict the instances of class 1.0(recall for this class is high)but it makes a lot of mistakes(very low presecion for class 1.0).So the model tries to predict non-oscar moovies with resut to predict few instances of class 0.0(recall of class 0.0 is very low). The presicion for class 0.0 is very high so if the model predict an instance of the dataset as a non-oscar moovie is very sure for this decesion. Accuracy is low , but maybe is influenced from the low recall for majority class.

In [ ]:
mean_df10=mean_of_folds(results10,'GNB')
mean10=plot_mean_of_folds(mean_df10)

In [ ]:
results11=KFold_fnc(SVC(kernel="linear"))

In [ ]:
plot_results(results11,'SVC with lenear kernel')

Pressicion ,Recall and F1-scores for class 0.0:
*   Pressicion values are high ,indicating accurate predisctions when the model identifies this class
*   Recall values are also high indicating thathe model captures most actual instances of this class
*   F1-scores are consistenly high showing a balance between presiscion and recall
---
Precesion ,Recall ,F1-scores for class 1.0:
*   Presicion is lower ,so teh model ha sa difficulty to predict accuarat ethe instances of this class.
*   Recall for this class is again low whicjh means taht the model can not distunguish well between the oscar moovies and no-oscar moovies.
*   F1-score is between 0.22-0.43 sowing a trade -off between recall and presicion for this class.
In summary the model identifies execellent the majority class ,but for the minority class does minor mistakes.



In [ ]:
mean_df11=mean_of_folds(results11,'SVC linear')
mean11=plot_mean_of_folds(mean_df11)

In [ ]:
results12=KFold_fnc(SVC(kernel="linear",class_weight={0:0.30,1:0.70}))

In [ ]:
plot_results(results12, 'SVM with linear,class_weights={0:0.30,1:0.70})')

Here we have also high presicion,recall,F1-scores high for class 0.0, indicating that the model can predict well for class 0.0.Presicion for class 1.0 ranges from low to moderate, indicating varying accuracy in identifying this class.Recall values for Class 1.0 also vary but generally range from 0.45 to 0.80, showing the model's ability to capture some of the actual instances ofthis class
F1-scores are good.Accuracy is also high across folds showing that the model can predict both classes well.

In [ ]:
mean_df12=mean_of_folds(results12,'SVM with linear,class_weights={0:0.30,1:0.70})')
mean12=plot_mean_of_folds(mean_df12)

In [ ]:
results13=KFold_fnc(SVC(kernel="poly",class_weight={0:0.30,1:0.70}))

Presicion for class 0.0 is really high indicating that when the model tries to identify class 0.0 is accurate to its predictions.ALso, recall for this class ranges from  0.98-1.00 meaning that the model can predict almost perfect the instances of class 0.0 .Still for class 1.0 we get a  low presicion and recall, meaning that the model is not able to predict the instances of class 1.0.Accuracy is really high because of the high scores for class 0.0. F1-scores for class 0.0 are  high but for class 1.0 are low,showing again the poor performance of the model for class 1.0.

In [ ]:
plot_results(results13, 'SVC(kernel="poly",class_weight={0:0.30,1:0.70})')

In [ ]:
mean_df13=mean_of_folds(results13,'SVM with linear,class_weights={0:0.30,1:0.70})')
mean13=plot_mean_of_folds(mean_df13)

In [ ]:
results14=KFold_fnc(SVC(kernel="rbf",class_weight={0:0.30,1:0.70}))

In [ ]:
plot_results(results14, 'SVM with rbf,class_weights={0:0.30,1:0.70})')

In [ ]:
mean_df14=mean_of_folds(results14,'SVM with rbf,class_weights={0:0.30,1:0.70})')
mean14=plot_mean_of_folds(mean_df14)

Presicion for class 0.0 is really high indicting that when the model tries to predict the class 0.0 is very accurate to its predictions. ALso, recall for class 0.0 is very high showing again that the model can find perfect all the instances of class 0.0. Presicion and recall for class 1.0 are 0 meaning the poor performance of the model to find and predict the instances for class 1.0. In fold 4 we get presicion for class 1.0 1.00 which means that the model is accurate to its predictions when tries to predict class 1.0 ,but we get a really low recall score.This shows that the model can only classifies few instances as class 1.0, being very confident for this desicion.F1-scores for clas 1.0 are mostly zero due to zero recall and presicion for this class. In the opposite F1-scores are really high for class 1.0. Accuracy is also high, but is influenced by majority class.

In [ ]:
results15=KFold_fnc(SVC(C=0.1,kernel="linear",gamma='scale'))

For class 0.0 we get really high scores of presicion,recall and F1-scores meaning that the model can identify and predict really perfect the instances of class 0.0. For class 1.0 still struggling to predict and identify them.

In [ ]:
plot_results(results15,'SVC(C=0.1,kernel="linear",gamma=scale')

In [ ]:
mean_df15=mean_of_folds(results15,'SVC(C=0.1,kernel="linear",gamma=scale')
mean15=plot_mean_of_folds(mean_df15)

In [ ]:
results16=KFold_fnc(SVC(C=0.1,kernel="linear",gamma='auto'))

Presicion, recall, and F1-scores(between 0.95-1.00) are high indicating that the model's performance for class 0.0 is almost perfect.For class 1.0 presicion, recall and F1-scores are low or zero, indicating the poor performance of the model to predict the class 1.0. For the folds where presisicon for class 1.0 is high we have a very low recall.This shows that the models struggles to find the instances of class 1.0,but when finds an oscar moovie ,is very confident that has predict well.This happens only for the 7% of the instances of class 1.0Accuracy is high due to high presicon and recall values for class 0.0.

In [ ]:
plot_results(results16,'SVC(C=0.1,kernel="linear",gamma=auto')

In [ ]:
mean_df16=mean_of_folds(results15,'SVC(C=0.1,kernel="linear",gamma=auto')
mean16=plot_mean_of_folds(mean_df16)

**Plotting the mean of metrics across folds**

In [ ]:
df_mean_folds_overall=pd.concat([mean_df1,mean_df2,mean_df3,mean_df4,mean_df5,mean_df6,mean_df7,mean_df8,mean_df9,mean_df10,mean_df11,mean_df12,mean_df13,mean_df14,mean_df15,mean_df16])
#print(df_mean_folds_overall)
plt.figure(figsize=(20,10))
indices=range(len(df_mean_folds_overall))
#print(indices)
bar_width=0.2

for i,col in enumerate(['1.0_precision_mean', '1.0_recall_mean', 'macro avg_f1-score_mean', 'accuracy_mean']):
  #print(i,col)
  plt.bar([ind+(i*bar_width) for ind in indices],df_mean_folds_overall[col],width=bar_width,label=col)

plt.xlabel('Names')
plt.xticks([ind + bar_width for ind in indices],df_mean_folds_overall['name'],rotation=90)
plt.legend()
plt.title('Mean of metrics across folds ')
plt.show()


After the 5-cross-fold validation we could say again, that the Desicion Tree Classifier and it's variations performs the best metrics.Presicion and recall for class 1.0 have not so mauch difference meaning that the model cam predict instances of class 1.0 correct.Accuracy and F1-scores are high.

##3.11 Undersampling

We try to get high scores on presicion and recall metrics.So we apply the undersampling technique for minority class(0:non-oscar values) hopping to get the wanted results. We apply undersampling only on training data beacuse after that we want to test the model on unseen imbalanced data(test-data).



In [ ]:
X_train

In [ ]:
from imblearn.under_sampling import RandomUnderSampler
#Initialize the RandomUnderSampler
under_sampler = RandomUnderSampler(sampling_strategy=0.5,random_state=42) #keep same size of non_oscar and oscar moovies
#try to predict class 1  with sampling_strategy=1
# Fit and undersample only the training data
X_train_resampled, y_train_resampled = under_sampler.fit_resample(X_train, y_train)


In [ ]:
type(X_train_resampled), type(y_train_resampled)

In [ ]:
df[df['oscar_winners'] == 1].shape

In [ ]:
X_train_resampled.shape, y_train_resampled.shape

In [ ]:
type(X_train_resampled), type(y_train_resampled)

### 3.10.1 Logistic Regression Model

####3.11.1.1 Simple Logistic Regression Model

In [ ]:
lr1_u=LogisticRegression()
lr1_u.fit(X_train_resampled,y_train_resampled)
y_pred=lr1_u.predict(X_test)
print("Classification report:\n",metrics.classification_report(y_test,y_pred))
lr_class_report_balanced=metrics.classification_report(y_test,y_pred, output_dict=True)
lr_class_report_balanced['name']='Logistic Regression(bd)'


####3.11.1.2 Logistic regression model with balanced class weights

In [ ]:
lr2_u=LogisticRegression(class_weight="balanced")
lr2_u.fit(X_train_resampled,y_train_resampled)
y_pred=lr2_u.predict(X_test)
print("Classification report:\n",metrics.classification_report(y_test,y_pred))
lr_class_report_balanced_balanced=metrics.classification_report(y_test,y_pred, output_dict=True)
lr_class_report_balanced_balanced['name']='Logistic Regression Balanced(bd)'


####3.11.1.3 Logistic Regression model with Fixed class weights

In [ ]:
lr3_u=LogisticRegression(class_weight={0:0.1, 1:0.9})
lr3_u.fit(X_train_resampled,y_train_resampled)
y_pred=lr3_u.predict(X_test)
print("Classification report:\n",metrics.classification_report(y_test,y_pred))
lr_class_report_class_weights_1_balanced=metrics.classification_report(y_test,y_pred, output_dict=True)
lr_class_report_class_weights_1_balanced['name']='Logistic Regression with custom class_weights_1(bd)'

In [ ]:
lr4_u=LogisticRegression(class_weight={0:0.5, 1:0.95})
lr4_u.fit(X_train_resampled,y_train_resampled)
y_pred=lr4_u.predict(X_test)
print("Classification report:\n",metrics.classification_report(y_test,y_pred))
lr_class_report_class_weights_2_balanced=metrics.classification_report(y_test,y_pred, output_dict=True)
lr_class_report_class_weights_2_balanced['name']='Logistic Regression with custom weights 2(bd) '


###3.11.2 Desicion Tree Classifier

####3.11.2.1 Simple Desicion Tree Classifier

In [ ]:
dt1_u=tree.DecisionTreeClassifier() #trying to get hiqh recall fro class oscar_winners
dt1_u.fit(X_train_resampled,y_train_resampled) #fit
y_pred=dt1_u.predict(X_test) #predict
print("Classification report\n",metrics.classification_report(y_test,y_pred))
dt_class_report_balanced=metrics.classification_report(y_test,y_pred,output_dict=True)
dt_class_report_balanced['name']='DT clf (bd)'


####3.11.2.2 Balanced Desicion Tree Classifier

In [ ]:
dt2_u=tree.DecisionTreeClassifier( class_weight='balanced') #trying to get hiqher recall and presiocn for class oscar_winners
dt2_u.fit(X_train_resampled,y_train_resampled)
y_pred=dt2_u.predict(X_test) #predict
print("Classification report:\n",metrics.classification_report(y_test,y_pred))
dt_class_custom_balanced_report_balanced=metrics.classification_report(y_test,y_pred, output_dict=True)
dt_class_custom_balanced_report_balanced['name']='DT balanced(bd)'
#dt_class_custom_weights1_report

####3.11.2.3 Weighted Class Desicion Tree Classifier

In [ ]:
dt3_u=tree.DecisionTreeClassifier( class_weight={0:0.1, 1:0.9}) #trying to get hiqher recall and presiocn for class oscar_winners
dt3_u.fit(X_train_resampled,y_train_resampled) #fit
y_pred=dt3_u.predict(X_test) #predict
print("Classification report:\n",metrics.classification_report(y_test,y_pred))
dt_class_custom_weights1_report_balanced=metrics.classification_report(y_test,y_pred, output_dict=True)
dt_class_custom_weights1_report_balanced['name']='DT custom_weights1 (bd)'
#dt_class_custom_weights1_report

In [ ]:
dt4_u=tree.DecisionTreeClassifier( class_weight={0:0.05, 1:0.95}) #trying to get hiqher recall and presiocn for class oscar_winners
dt4_u.fit(X_train_resampled,y_train_resampled)
y_pred=dt4_u.predict(X_test) #predict
print("Classification report:\n",metrics.classification_report(y_test,y_pred))
dt_class_custom_weights2_report_balanced=metrics.classification_report(y_test,y_pred, output_dict=True)
dt_class_custom_weights2_report_balanced['name']='DT custom_weights2 (bd)'
#dt_class_custom_weights1_report

###3.11.3 KNN

####3.11.3.1  KNeighbors Classifier

In [ ]:
from sklearn.neighbors import KNeighborsClassifier as KN
knn_clf1_u=KN(weights='uniform') #trying to get hiqh recall fro class oscar_winners
knn_clf1_u.fit(X_train_resampled,y_train_resampled) #fit
print("Classification report\n",metrics.classification_report(y_test,y_pred))
knn_class_report_uniform_balanced=metrics.classification_report(y_test,y_pred, output_dict=True)
knn_class_report_uniform_balanced['name']='KNN_uniform (bd)'
#knn_class_report_uniform

####3.11.3.2Kneigbors Classifier with weigts='distance'

In [ ]:
from sklearn.neighbors import KNeighborsClassifier as KN
knn_clf2_u=KN(weights='distance') #trying to get hiqh recall fro class oscar_winners
knn_clf2_u.fit(X_train_resampled,y_train_resampled) #fit
print("Classification report\n",metrics.classification_report(y_test,y_pred))
knn_class_report_dist_balanced=metrics.classification_report(y_test,y_pred, output_dict=True)
knn_class_report_dist_balanced['name']='KNN_distance (bd)'
#knn_class_report_dist

###3.11.4 GaussianNB

In [ ]:
from sklearn.naive_bayes import GaussianNB as GNB
gnb_clf_u=GNB() #trying to get hiqh recall fro class oscar_winners
gnb_clf_u.fit(X_train_resampled,y_train_resampled) #fit
y_pred=gnb_clf_u.predict(X_test)
#print(y_pred)
print("Classification report\n",metrics.classification_report(y_test,y_pred))
gnb_class_report_balanced=metrics.classification_report(y_test,y_pred, output_dict=True)
gnb_class_report_balanced['name']='GNB(bd)'
#gnb_class_report

###3.11.5 Support Vector Machines

####3.11.5.1 SVM with linear kernel

In [ ]:
from sklearn import svm
svm1_u=svm.SVC(kernel="linear")
svm1_u.fit(X_train_resampled,y_train_resampled)
#print(svm.predict(X_test))
print("CLassification report\n",metrics.classification_report(y_test,y_pred))
svm_class_report_balanced=metrics.classification_report(y_test,y_pred,output_dict=True)
svm_class_report_balanced['name']='SVM (bd)'
#plt.show(confusionMatrix_and_roc(svm1_u,X_test,y_test))
#svm_class_report


####3.11.5.2 SVM algorithm with lenear kernel  and fixed class weights

In [ ]:
from sklearn.svm import SVC
svm2_u=SVC(kernel="linear",class_weight={0:0.30,1:0.70})
svm2_u.fit(X_train_resampled,y_train_resampled)
#print(svm.predict(X_test))
print("CLassification report\n",metrics.classification_report(y_test,y_pred))
svm_class_report_weights_balanced=metrics.classification_report(y_test,y_pred,output_dict=True)
svm_class_report_weights_balanced['name']='SVM fixed weights (bd)'


####3.11.5.3 SVM algorithm with polynomial kernel and fixed class weights

In [ ]:
svm3_u=SVC(kernel="poly",class_weight={0:0.30,1:0.70})
svm3_u.fit(X_train_resampled,y_train_resampled)
#print(svm.predict(X_test))
print("CLassification report\n",metrics.classification_report(y_test,y_pred))
svm_class_report_weights_poly_balanced=metrics.classification_report(y_test,y_pred,output_dict=True)
svm_class_report_weights_poly_balanced['name']='SVM class_weights, polynomial kernel(bd)'

####3.11.5.4 SVM algorithm with rbf kernel and fixed class weights

In [ ]:
svm4_u=SVC(kernel="rbf",class_weight={0:0.30,1:0.70})
svm4_u.fit(X_train_resampled,y_train_resampled)
#print(svm.predict(X_test))
print("CLassification report\n",metrics.classification_report(y_test,y_pred))
svm_class_report_weights_rbf_balanced=metrics.classification_report(y_test,y_pred,output_dict=True)
svm_class_report_weights_rbf_balanced['name']='SVM weight_class and rbf kernel in balanced dstaset'
#svm_class_report_weights

####3.11.5.5 SVM algorithm wit linear kernel ,C=0 1, gamma='scale'

In [ ]:
svm5_u=SVC(C=0.1,kernel="linear",gamma='scale')
svm5_u.fit(X_train_resampled,y_train_resampled)
#print(svm.predict(X_test))
print("CLassification report\n",metrics.classification_report(y_test,y_pred))
svm_class_report_weights_gamma_scale_balanced=metrics.classification_report(y_test,y_pred,output_dict=True)
svm_class_report_weights_gamma_scale_balanced['name']='SVM with C=0.1,gamma=scale,kernel=linear(bd)'

####3.11.5.6 SVM algoriyhm with linear kernel ,C=0.1, gamma='auto

In [ ]:
svm6_u=SVC(C=0.1,kernel="linear",gamma='auto')
svm6_u.fit(X_train,y_train)
#print(svm.predict(X_test))
print("CLassification report\n",metrics.classification_report(y_test,y_pred))
svm_class_report_weights_gamma_auto_balanced=metrics.classification_report(y_test,y_pred,output_dict=True)
svm_class_report_weights_gamma_auto_balanced['name']='SVM with C=0.1 and gamma=auto(bd)'


###3.11.7 Collect all metrics in a dataset and plot

In [ ]:
undersampling_df=pd.DataFrame([flatten_dict(lr_class_report_balanced),
                               flatten_dict(lr_class_report_balanced_balanced),
                               flatten_dict(lr_class_report_class_weights_1_balanced),
                               flatten_dict(lr_class_report_class_weights_2_balanced),

                               flatten_dict(dt_class_report_balanced),
                               flatten_dict(dt_class_custom_balanced_report_balanced),
                               flatten_dict(dt_class_custom_weights1_report_balanced),
                               flatten_dict(dt_class_custom_weights2_report_balanced),

                               flatten_dict(knn_class_report_uniform_balanced),
                               flatten_dict(knn_class_report_dist_balanced),

                               flatten_dict(gnb_class_report_balanced),

                               flatten_dict(svm_class_report_balanced),
                               flatten_dict(svm_class_report_weights_balanced),
                               flatten_dict(svm_class_report_weights_poly_balanced),
                               flatten_dict(svm_class_report_weights_rbf_balanced),
                               flatten_dict(svm_class_report_weights_gamma_scale_balanced),
                               flatten_dict(svm_class_report_weights_gamma_auto_balanced),
                               ])
#undersampling_df

In [ ]:
#plt.figure(figsize=(9000,58))
undersampling_df.plot(x="name",y=['1.0_precision','1.0_recall','macro avg_f1-score','accuracy'], kind="bar")
plt.title('Classification Metrics Across Models')
plt.legend(loc='lower left',bbox_to_anchor=(1,0.5))
plt.show()

From the figure,we observe that the Logistic Regression algorithm,the Desicion Tree Classifier with custom weights_1 and the Logistic Regression with balance class weights have the best results. The precision of class 1 is the highest, and also has a high recall.

In [ ]:
#combine 3 best classifiers
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import cross_validate
clf1=LogisticRegression()
clf2=tree.DecisionTreeClassifier( class_weight={0:0.1, 1:0.9})
clf3=LogisticRegression(class_weight='balanced')
eclf2=VotingClassifier(estimators=[('lr',clf1),('dt_w',clf2),('lr_b',clf3)],voting='hard')
eclf2.fit(X_train,y_train)
pred=eclf2.predict(X_test)
scores = cross_validate(eclf2, X_train, y_train, scoring=('accuracy','precision','recall'), cv=5)
print("Accuracy: %0.2f  [%s]" % (scores['test_accuracy'].mean(), 'Enseble'))
print("Presicion: %0.2f  [%s]" % (scores['test_precision'][2].mean(), 'Enseble'))
print("Recall: %0.2f  [%s]" % (scores['test_recall'][2].mean(), 'Enseble'))
#print(scores['test_recall'])

The voting classifier gives a high presicion ,along with a high recall,meaning that the voting classifier has strong abbillieties to find all moovies and classifies them correct.

#4.Data Cleaning of test set

##4.1 Load Datatset

In [ ]:
df_test= pd.read_excel('movies_test _anon.xlsx') #load datatset
df_test

#df = pd.read_excel('movies.xlsx') #load datatset
#df

##4.2 Datatype of columns and fix columns

In [ ]:
df_test.dtypes

In [ ]:
df_test.columns


In [ ]:
#fixed type of columns
new_columns=[col.strip().replace('  ',' ').replace(' ','_').lower() for col in df_test.columns]
df_test.columns=new_columns

In [ ]:
#check if done
df_test.columns

##4.3 Remove columns

We drop columns ```'distributor','imdb_rating', 'imdb_vs_rt_disparity', 'film',``` because we dont have them on training process

In [ ]:
df_test.isna().sum()

In [ ]:
#drop columns
df_test.drop(['distributor','imdb_rating', 'imdb_vs_rt_disparity','film'],axis=1, inplace=True)

##4.4 Handling with missing values on categorical variables

Here we can just drom the null values because we want to predict for every moovie in the datatset if it wins an osacr or not.

In [ ]:
df_test.isnull().sum() #see where do we have null values /column

We see that we have 4 missing values in column ```Genre``` and 29 misiing values in column ```Primary Genre```. We want to see if the missing values of column ```Primary Genre``` is in the same position with the misisng values of ```Genre``` in order to complete the missing values in column ```Genre```

In [ ]:
df_test[df_test['genre'].isnull() & df_test['primary_genre'].isnull()]
#df[df['oscar_detail'].isnull() | df['oscar_winners'].isnull()]

Now we camn drop column ```primary_genre```

In [ ]:
df_test.drop(['primary_genre'], axis=1, inplace=True)

We substitute missing values and rows with '-' of column ```genre``` with the most frequent genre value in the dataset.

In [ ]:
#find the most frequent value for 'genre' in the dataset
df_test.value_counts('genre')

In [ ]:
df_test[df_test['genre'].isnull()]


In [ ]:
df_test['genre']=df_test['genre'].replace(np.nan,'comedy') #replace with th emost frequent value in the df

In [ ]:
df_test['genre']=df_test['genre'].replace('-','comedy')

In [ ]:
df_test[df_test['genre'].isnull()] #check if done

Do the same for column ```script_type```

In [ ]:
df_test[df_test['script_type'].isnull()]

In [ ]:
df_test['script_type']=df_test['script_type'].replace(np.nan,'original screenplay') #replace with the most frequent value in the df

In [ ]:
df_test[df_test['script_type'].isnull()] #check if done

See also were do we have values like '-'

In [ ]:
df_test[df_test['script_type']=='-']

##4.5 Handling with values like '-'

In [ ]:
df_test.head(20) #see the type of our cells fro the first 20 lines

In [ ]:
df.tail(20)

In [ ]:
df_test.isna().sum()

In [ ]:
df_test[~df_test['rotten_tomatoes_critics'].apply(np.isreal) |~df_test['metacritic_critics'].apply(np.isreal) ] #check if done

In [ ]:
columns_to_replace_with_mean=['metacritic_critics','rotten_tomatoes_critics',
                              'average_critics','metacritic_audience','rotten_tomatoes_vs_metacritic_deviance',
                              'average_audience','audience_vs_critics_deviance'
                              ]

In [ ]:
df[columns_to_replace_with_mean]=df[columns_to_replace_with_mean].astype(float) #convert to float

In [ ]:
for col in df_test.columns:
    df_test[col]=df_test[col].map(replace_str_with_nan)

In [ ]:
for col in columns_to_replace_with_mean:
  mean_col=df_test[col].mean() #find mean for every column
  df_test[col].fillna(mean_col, inplace=True) #fill na with mean

In [ ]:
df_test.isna().sum() #check if done for this spesific columns

##4.5 Handling with columns that has data with symbol %

In [ ]:
df_test.dtypes


In [ ]:
df_test['of_gross_earned_abroad'].head(5)

In [ ]:
columns_to_convert=['of_gross_earned_abroad','budget_recovered_opening_weekend','budget_recovered']
df_test[columns_to_convert]=df_test[columns_to_convert].astype(str)


In [ ]:
#make regex to extract ('%') of every column that has this symbol in it's values and keep only the real number without the symbol
df_test['of_gross_earned_abroad']=df_test['of_gross_earned_abroad'].str.extract(r'(\d+\.\d+)')
df_test['budget_recovered_opening_weekend']=df_test['budget_recovered_opening_weekend'].str.extract(r'(\d+\.\d+)')
df_test['budget_recovered']=df_test['budget_recovered_opening_weekend'].str.extract(r'(\d+\.\d+)')

In [ ]:
df_test[columns_to_convert].isna().sum()

In [ ]:
df_test.dtypes

Replace '-' for column ```of_gross_earned_abroad``` with na, then replace na with the mean.

In [ ]:
df_test['of_gross_earned_abroad']=df_test['of_gross_earned_abroad'].replace('-',np.nan)

In [ ]:
#convert into datatype float
df_test['of_gross_earned_abroad']=df_test['of_gross_earned_abroad'].astype(float)

In [ ]:
mean=df_test['of_gross_earned_abroad'].mean()
df_test['of_gross_earned_abroad']=df_test['of_gross_earned_abroad'].fillna(mean)

In [ ]:
#check type of every column
df_test.dtypes

##4.6 Handling with columns opening_weekend,domestic_gross,foreign_gross,worldwide_gross,budget_($million)

We see that the columns ```'opening_weekend_($million)','domestic_gross_($million)','foreign_gross_($million)','worldwide_gross_($million)'``` are similar with ```opening_weekend,domestic_gross,foreign_gross,worldwide_gross```. So we drop them.

In [ ]:
df_test.drop(['opening_weekend_($million)','domestic_gross_($million)','foreign_gross_($million)','worldwide_gross_($million)'],axis=1,inplace=True)

We see id the columns ```opening_weekend,domestic_gross,foreign_gross,worldwide_gross``` have symbols like '-',in order to replace them to do later the converiso to float.

In [ ]:
df_test[~df_test['opening_weekend'].map(np.isreal)]

In [ ]:
df_test[~df_test['domestic_gross'].map(np.isreal)]

In [ ]:
df_test[~df_test['foreign_gross'].map(np.isreal)]

In [ ]:
df_test[~df_test['worldwide_gross'].map(np.isreal)]

We see that we have numbers with commas.So, we substitute the commas with '' nothing

In [ ]:
df_test['foreign_gross']=df_test['foreign_gross'].map(replace_comma)

In [ ]:
df_test[~df_test['foreign_gross'].map(np.isreal)] #check if done

We also replace na values with the mean of the columns

In [ ]:
#opening_weekend,domestic_gross,foreign_gross,worldwide_gross
mean=df_test['opening_weekend'].mean()
df_test['opening_weekend']=df_test['opening_weekend'].fillna(mean)

In [ ]:
mean=df_test['domestic_gross'].mean()
df_test['domestic_gross']=df_test['domestic_gross'].fillna(mean)

In [ ]:
mean=df_test['worldwide_gross'].mean()
df_test['worldwide_gross']=df_test['worldwide_gross'].fillna(mean)

For column ```foreign_gross``` we have values like '-'. So we replace these values with na.

In [ ]:
df_test['foreign_gross']=df_test['foreign_gross'].apply(replace_str_with_nan)

In [ ]:
df_test.dtypes

In [ ]:
df_test['foreign_gross']=df_test['foreign_gross'].astype(float)

In [ ]:
mean=df_test['foreign_gross'].mean()
df_test['foreign_gross']=df_test['foreign_gross'].fillna(mean)

##4.7 Handling with release_date_us

In [ ]:
df_test['release_date_(us)'].head(10)

Convert ```release_date_us``` to date format. we see that we have date values like Jan 19, 2018 (limited), Jan 26, 2018  ','Jan 19, 2018, so we replace them with Jan 19, 2018.

In [ ]:
#got error Unknown string format: Jan 19, 2018 (limited), Jan 26, 2018   present at position 110
#regex to find this patterns

df_test['release_date_(us)'].iloc[110]=df_test['release_date_(us)'].iloc[110].replace('Jan 19, 2018 (limited), Jan 26, 2018  ','Jan 19, 2018')
df_test['release_date_(us)'].iloc[176]=df_test['release_date_(us)'].iloc[176].replace('Sep 16, 2020 (Netflix)','Sep 16, 2020')
df_test['release_date_(us)'].iloc[288]=df_test['release_date_(us)'].iloc[288].replace('Apr 27, 2018 (limited), May 18, 2018  ','Apr 27, 2018')
df_test['release_date_(us)'].iloc[338]=df_test['release_date_(us)'].iloc[338].replace('Jul 10, 2020 (Apple TV+)','Jul 10, 2020')
df_test['release_date_(us)'].iloc[415]=df_test['release_date_(us)'].iloc[415].replace('September 30, 2021 (limited release), October 21, 2021 (VOD)','September 30, 2021')
df_test['release_date_(us)'].iloc[497]=df_test['release_date_(us)'].iloc[497].replace('Sep 23, 2020 (Netflix)','Sep 23, 2020')
df_test['release_date_(us)'].iloc[507]=df_test['release_date_(us)'].iloc[507].replace('November 12, 2021 (limited release), November 19, 2021 (wide release)','November 12, 2021')
df_test['release_date_(us)'].iloc[537]=df_test['release_date_(us)'].iloc[537].replace('Jun 5, 2020 (limited), Jun 12, 2020 (VOD)', 'Jun 5, 2020')
df_test['release_date_(us)'].iloc[562]=df_test['release_date_(us)'].iloc[562].replace('Apr 24, 2020 (Netflix)','Apr 24, 2020')


In [ ]:
df_test['release_date_(us)']=pd.to_datetime(df_test['release_date_(us)'])
df_test['release_date_(us)']=pd.to_datetime(df_test['release_date_(us)'],format="%m/%d/%Y") #convert to 'YY-MM-DD' format
df_test['release_date_(us)']

In [ ]:
df_test['release_date_(us)'].head(10)

In [ ]:
#keep only the year
df_test['release_date_(us)']=df_test['release_date_(us)'].dt.year

In [ ]:
df_test['release_date_(us)'].mode() #fin dthe most frequent

In [ ]:
df_test['release_date_(us)']=df_test['release_date_(us)'].replace(np.nan,pd.Timestamp('2018').year)

In [ ]:
df_test['release_date_(us)'].isna().sum() #check if done

In [ ]:
df_test['release_date_(us)']=df_test['release_date_(us)'].astype(int) #convert to type int

##4.8 Columns budget_($million), budget_recovered,budget_recovered_opening_weekend

See where do we have null values

In [ ]:
df_test[df_test['budget_($million)'].isnull()] #see which lines in column budget_($million) are null

Replace na values with the mean of the col

In [ ]:
mean_col=df_test['budget_($million)'].mean() #find mean
df_test['budget_($million)'].fillna(mean_col, inplace=True) #fill na with mean

In [ ]:
df_test[df_test['budget_($million)'].isnull()] #check if done

In [ ]:
df_test[df_test['budget_recovered'].isnull()] #the same goes with 'budget_recovered' column

In [ ]:
df_test.dtypes

In [ ]:
df_test['budget_recovered']=df_test['budget_recovered'].astype(float)

In [ ]:
mean_col=df_test['budget_recovered'].mean() #find mean
df_test['budget_recovered'].fillna(mean_col, inplace=True) #fill na with mean
df_test[df_test['budget_recovered'].isnull()] #check if done

In [ ]:
df_test['budget_recovered_opening_weekend']=df_test['budget_recovered_opening_weekend'].astype(float)

In [ ]:
mean_col=df_test['budget_recovered_opening_weekend'].mean() #find mean
df_test['budget_recovered_opening_weekend'].fillna(mean_col, inplace=True) #fill na with mean
df_test[df_test['budget_recovered_opening_weekend'].isnull()] #check if done

In [ ]:
df_test.dtypes

In [ ]:
df_test.head(10)

In [ ]:
df_test.tail(20)

In [ ]:
df_test.isna().sum()

In [ ]:
df_test.dtypes

In [ ]:
df_test['rotten_tomatoes_audience']=df_test['rotten_tomatoes_audience'].astype(float)

##4.9 Convert categorical data into multi-hot encoding

Στόχος είναι να κάνουμε multi-hot encoding για την στήλη 'genre'.

####4.9.1 Column genre

In [ ]:
df_test.dtypes

In [ ]:
df_test.genre

In [ ]:
df_test.genre.unique() #keep only the unique values of column 'script_type'

In [ ]:
df_test['genre']=df_test['genre'].str.strip().str.lower() #with strip function ,erase tthe ' ' before or after the quotes and then convert to lower_case
df_test['genre'].unique() #check if done

In [ ]:
columns_of_genre_1=df_test['genre'].str.split(', ').explode().unique() #check if done and check also the unique genre type
df4=pd.DataFrame(columns=columns_of_genre_1)
df4
columns_of_genre_1

Replace thriler-thriller,mystery & thriller- mystery/thriller, music-musical,sport-sports, horror/mystery-horror, mystery, fanstasy-fantasy, black comedy-comedy

In [ ]:
df_test[df_test['genre'].str.contains(r'\bmystery & thriller\b')] #see where 'sport' is containing along with other features

In [ ]:
df_test['genre']=df_test['genre'].str.replace(r'\bmystery & thriller\b', 'mystery/thriller')#replace

In [ ]:
df_test['genre'].str.contains(r'\bblack comedy\b')
df_test['genre']=df_test['genre'].str.replace(r'\bblack comedy\b', 'comedy')#replace

In [ ]:
df_test['genre']=df_test['genre'].str.replace(r'\bmystery/thriller\b', 'mystery, thriller')#replace

In [ ]:
df_test[df_test['genre'].str.contains(r'\bhorror/mystery\b')]

In [ ]:
df_test['genre']=df_test['genre'].str.replace(r'\bhorror/mystery\b','horror, mystery')

In [ ]:
df_test[df_test['genre'].str.contains(r'\fanstasy\b')]
df_test['genre']=df_test['genre'].str.replace(r'\fanstasy\b','fantasy')

In [ ]:
df_test[df_test['genre'].str.contains(r'\bsport\b')]

In [ ]:
df_test['genre']=df_test['genre'].str.replace(r'\bsport\b','sports')

In [ ]:
df_test[df_test['genre'].str.contains(r'\bthriler\b')] #see where 'thriler' is containing along with other features
df_test['genre']=df_test['genre'].str.replace(r'\bthriler\b','thriller') #replace with 'thriller'

In [ ]:
df_test[df_test['genre'].str.contains(r'\bmusic\b')] #see where 'music' is containing along with other features


In [ ]:
df_test['genre']=df_test['genre'].str.replace(r'\bmusic\b','musical')

In [ ]:
df_test[df_test['genre'].str.contains(r'sci-fi\. thriller')] #see where ''sci-fi\. thriller' is containing

Χρησιμοποιούμε τα unique values of genre μεμονωμένα και φτιάχνουμε ένα dataframe με αυτές τις στήλες

In [ ]:
columns_of_genre_2=df_test['genre'].str.split(', ').explode().unique() #split genre and get the unique values from everry line in column genre
df4=pd.DataFrame(columns=columns_of_genre_2) #make a dataframe with columns_of_genre
#df1
columns_of_genre_2 #see all the uniqye values

In [ ]:
df_test.dtypes

In [ ]:
df_test=pd.concat([df_test,df4]) #concat two dataframes

In [ ]:
df_test['year']=df_test['year'].astype(int) #μου αλλαζει το year οταν κανω το concat οπότε το μετατρέπω ξανά σε int

In [ ]:
df_test.dtypes

In [ ]:
for i,row in enumerate (df_test['genre']): #loop all the lines of column 'genre'
  l=(row.split(', ')) #split the line to get the different types of 'genre'
  for item in l:
    #print(item)
    df_test.loc[i,item]=1 #put 1 in the df where the type of genre is appeared

In [ ]:
df_test[columns_of_genre_2]=df_test[columns_of_genre_2].fillna(0) #put 0 to all the other lines

In [ ]:
df_test #check if done

In [ ]:
#drop column genre
df_test.drop(['genre'], axis=1, inplace=True)

####4.10.2 Column script_type


Στόχος είναι να κάνουμε multi-hot encoding για την στήλη 'script_type'.

In [ ]:
df_test['script_type']=df_test['script_type'].str.strip().str.lower() #erase ' ' before or after quotes  and then to lower_case
df_test['script_type'].unique() #see unique values

In [ ]:
columns_of_script_type=df_test['script_type'].str.split(', ').explode().unique() #split by comma and get the unique values of column 'script_type'
columns_of_script_type

In [ ]:
df6=pd.DataFrame(columns=columns_of_script_type) #make a dataframe with the unique values of column 'script_type'
df6

In [ ]:
df_test=pd.concat([df_test,df6]) #concat the 2 datafames
df_test

In [ ]:
for i,row in enumerate (df_test['script_type']):
  l=(row.split(', ')) #split every line by comma
  for item in l: #where the item is appeared put 1
    #print(item)
    df_test.loc[i,item]=1

In [ ]:
df_test[columns_of_script_type]=df_test[columns_of_script_type].fillna(0) #put to every other column 0

In [ ]:
df_test.dtypes #check type of every column

In [ ]:
df_test

In [ ]:
#drop column script type
df_test.drop(['script_type'], axis=1, inplace=True)

In [ ]:
col=['original screenplay','sequel','remake','adaptation','based on a true story','prequel','semi-sequel']
df_test[col] = df_test[col].astype(float) #convert to float type

In [ ]:
df_test.dtypes

In [ ]:
df_test.isna().sum()

In [ ]:
df.columns


In [ ]:
df_test.columns

In [ ]:
df_test.shape,df.shape

In [ ]:
df_test=df_test.drop('year',axis=1)

In [ ]:
df_test.shape,df.shape

In [ ]:
df.columns

In [ ]:
df_test.columns

#5.Final predictions

In [ ]:
X_new = df_test
#pred=eclf2.predict(X_new)
#print(pred)


Scaling data in the test set(we have forgotten to do that,so we got a really high value of oscar-winners moovies)

In [ ]:
#Standardization
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

scaler.fit(X_new) #let scaler to fit to X_train #find the mean and std of the data
X_new=scaler.transform(X_new) #let scaler to transform the data


In [ ]:
pred=eclf2.predict(X_new)
print(pred)

In [ ]:
results_df=pd.DataFrame({'id': df_test['id'], 'prediction':pred})


In [ ]:
results_df

In [ ]:
results_df['prediction'].value_counts()

In [ ]:
#uncomment to download the final results
#from google.colab import files
#results_df.to_csv('predictions_results.csv', index=False)
#files.download('predictions_results.csv')


# 6.Clustering

In [ ]:
X=df.drop(['oscar_winners'], axis=1)
y=df['oscar_winners']
X.shape, y.shape

In [ ]:
#Standardization
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

scaler.fit(X) #let scaler to fit to X_train #find the mean and std of the data
X=scaler.transform(X) #let scaler to transform the data



We drop all the features of categorical variables in order to do the correlation with the other features.

In [ ]:
df.columns


In [ ]:
df.drop(['period', 'action', 'western', 'horror', 'musical',
       'sci-fi', 'family', 'comedy', 'crime', 'drama', 'animation', 'sports',
       'war', 'thriller', 'fantasy', 'adventure', 'romantic comedy', 'romance',
       'documentary', 'biography', 'history', 'mystery', 'adaptation',
       'remake', 'original screenplay', 'sequel', 'based on a true story',
       'remake / reboot', 'prequel'],axis=1,inplace=True)

##K-means Clustering

###Evaluate Kmeans

In [ ]:
from sklearn import metrics
from sklearn.cluster import KMeans
from collections import defaultdict

rand_scores=defaultdict(list) #create a default dict where the default value for every key is a list
for k in range (2, 11):
  km =KMeans(n_clusters=k, n_init=100, random_state=42)
  km.fit(X)
  pred=km.labels_
  df['pred']=pred
  #sns.histplot(data=df, x='pred', hue='oscar_winners')
  rand_scores['k'].append(k)
  rand_scores['rand_score'].append(metrics.adjusted_rand_score(pred,y))
  rand_scores['MI'].append(metrics.mutual_info_score(y,pred)) #mutual info score
  rand_scores['AMI'].append(metrics.adjusted_mutual_info_score(y,pred)) #adjuasted mutual info score
  rand_scores['NMI'].append(metrics.normalized_mutual_info_score(y,pred)) #normalized mutual info score
  rand_scores['Homogeneity'].append(metrics.homogeneity_score(y,pred))
result_df=pd.DataFrame(rand_scores)
result_df.plot(x='k')


From the plot we see that when we increase the number of clusters the scores dicrease.That's why these metrics try to match the actual predictions with the clusters predictions ignoring the permutations. So, maybe it is more accurate to see every metric for  a specific number of clusters. Also we do not have a clear result for the clustering

In [ ]:
df.columns

###PCA and Kmeans

We try the PCA algorithm

In [ ]:
from sklearn import metrics
from sklearn.cluster import KMeans
from collections import defaultdict
from sklearn.decomposition import PCA


scores=defaultdict(list) #create a default dict where the default value for every key is a list
for comp in range (3, 20):
  pca=PCA(n_components=comp)
  X_norm=pca.fit_transform(X)
  km =KMeans(n_clusters=3, n_init=100, random_state=42)
  km.fit(X_norm)
  pred=km.labels_
  df['pred']=pred
  #sns.histplot(data=df, x='pred', hue='oscar_winners')
  scores['comp'].append(comp)
  scores['rand_score'].append(metrics.adjusted_rand_score(pred,y))
  scores['MI'].append(metrics.mutual_info_score(y,pred)) #mutual info score
  scores['AMI'].append(metrics.adjusted_mutual_info_score(y,pred)) #adjuasted mutual info score
  scores['NMI'].append(metrics.normalized_mutual_info_score(y,pred)) #normalized mutual info score
  scores['Homogeneity'].append(metrics.homogeneity_score(y,pred))
result_df=pd.DataFrame(scores)
result_df.plot(x='comp')



From the plot we see that applying k-means with 3 cluster in the dataset with different size of features we do not have better results for clustering

In [ ]:
from sklearn import metrics
from sklearn.cluster import KMeans

km =KMeans(n_clusters=2, n_init=100, random_state=42)
km.fit(X)
pred=km.labels_
df['pred']=pred
sns.histplot(data=df, x='pred', hue='oscar_winners')

We see that in the cluster 1 belongs all the moovies, that have not received an oscar. In cluster 0 we see that there are some moovies that has won oscar and some non-oscar moovies. So ,probably with 2 clusetrs is not the ideal clustering

More specifically, in the initial dataset we have 58 moovies that have received oscar and in clusetr 0 we have cluster only the 56/58 and rest that has not received an oscar.

In [ ]:
df[pred==0]['oscar_winners'].value_counts() #56 /58 moovies right

We see that in cluster 1 we have mostly non-oscar values

In [ ]:
df[pred==1]['oscar_winners'].value_counts()

We try to map the results from the clusters in order to compare them with the actual value y .So cluster 1-> all non-oscar moovies,cluster 0->both (non-oscar and osacr moovies)

In [ ]:
from sklearn.metrics import classification_report
class_preds=[1 if c==0 else 0 for c in pred ]
print(classification_report(y,class_preds))

We see that this clustering has a low accuracy.Presicion for class 0.0 is high indicating that the cluster algorithm can classify all the non-oscar moovies.Recall is lower indicating that the algorithm can not predict the class 0.0 very well. Presision for class 0.0 is low,meaning that the algorithm tries to find the non-oscar moovies,making a lot of mistakes on class 0.0.So, we see that the recall for class 0.0 is low.

We see that the algorithm makes a lot of  mistakes on cluster 1. We try to cluster(again) only the cluster 0

In [ ]:
X1=X[pred==0]
#X1.shape
y1=y[pred==0] #actual classes for data, where the pred==0
df1=df[df['pred']==0].copy() #make a new dataframe with data that belongs only on cluster 0
km1=KMeans(n_clusters=2, n_init=100, random_state=42)
km1.fit(X1)
pred1=km1.labels_
df1['pred1']=pred1
sns.histplot(data=df1, x='pred1', hue='oscar_winners')

We try to map the results from the clusters in order to compare them with the actual value y .So cluster 1-> all non-oscar moovies,cluster 0->both (non-oscar and osacr moovies)

In [ ]:
from operator import length_hint
class_pred1=[0 if c==1 else 1 for c in pred1 ]
length_hint(class_pred1)

We try to make a hierarchical clustering
Size of initial data 1400. -->
1rst layer:Kmean(2 clusters).Cluster_0:oscar and non_oscar ,
Cluster_1: mostly non-oscar -->2nd layer:
Kmean(2 clusters) on cluster_0 only. cluster0_0: oscar nad non-oscar, cluster 0_1: non-oscar







In [ ]:
#try to concat pred1 with initial pred
pred_final =np.array([False if c!=0 else 5 for c in pred])#if we are in cluster 1,False , else 5(not True because it interprents it as 1,whicj in this case is wrong )

pred_final[pred_final==5] = class_pred1
print(classification_report(y,pred_final))


From the classification report we see that we have a better model with higher accuracy

For these clusters we  try to find the correlation with other features

In [ ]:
for col in df.columns:
  sns.histplot(data=df, x=col, hue=df['pred'], alpha=0.6, palette='tab10')
  plt.title(col)
  plt.show()

From the histplots we can see that moovies with no oscars have high rotten_tomatos_critics,metacritic critics,average critics,rotten_tomatoes_audience,metacritic_audience,average_audience,audience_vs_critic_deviance. Moovies with oscar have lower scores for these features.

### Kmeans and gridsearch

We try to improve kmean clustering performance

In [ ]:
from sklearn.metrics import adjusted_rand_score,adjusted_mutual_info_score,mutual_info_score,homogeneity_score,normalized_mutual_info_score,fowlkes_mallows_score
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

param_grid={
    'kmeans__n_clusters':[2,3,4,5],
    'kmeans__init':['k-means++','random'],
    'kmeans__n_init':[100],
    'kmeans__max_iter':[500]
}
scoring={
    'rand_score':make_scorer(adjusted_rand_score),
    'adjusted_mutual_info_score':make_scorer(adjusted_mutual_info_score),
    'fowlkes_mallows':make_scorer(fowlkes_mallows_score),
    'mutual_info_score':make_scorer(mutual_info_score),
    'homogeneity_score':make_scorer(homogeneity_score),
    'normalized_mutual_score':make_scorer(normalized_mutual_info_score)
}
pipeline=Pipeline([('scaler', StandardScaler()), ('kmeans',KMeans())])
grid_search=GridSearchCV(pipeline, param_grid, scoring=scoring,refit='rand_score',cv=5)
grid_search.fit(X,y)
print('Best Parameters for kmean',grid_search.best_params_)


We see that the grid search algorithm gives as best parameters {'kmeans__init': 'random', 'kmeans__max_iter': 500, 'kmeans__n_clusters': 4, 'kmeans__n_init': 100}. So we make a kmean algorithm to evaluate again the clustering

In [ ]:
km =KMeans(n_clusters=4, n_init=100, random_state=42,max_iter=500)
km.fit(X)
pred=km.labels_
df['pred']=pred
sns.histplot(data=df, x='pred', hue='oscar_winners')

Try to map the clusters with the actual values of column oscar_winners

In [ ]:
pred=[1 if (c==1 or c==2 or c==3) else 0 for c in pred ]
print(classification_report(y,pred))

From the classification report we see that we don't get better results for clustering

## HCA

###Evaluate HCA

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score,adjusted_mutual_info_score,mutual_info_score,homogeneity_score,normalized_mutual_info_score
from sklearn import metrics
from collections import defaultdict


scores = defaultdict(list)  # create a default dict where the default value for every key is a list
n_clusters = [2, 3, 4, 5]
linkage = ['ward', 'complete', 'average', 'single']

for c in n_clusters:
    for link in linkage:
        aggl = AgglomerativeClustering(n_clusters=c, linkage=link)
        aggl.fit(X)
        pred = aggl.labels_
        df['pred'] = pred
        scores['c'].append(c)
        scores['link'].append(link)
        scores['rand_score'].append(metrics.adjusted_rand_score(pred, y))
        scores['MI'].append(metrics.mutual_info_score(y, pred))
        scores['AMI'].append(metrics.adjusted_mutual_info_score(y, pred))
        scores['NMI'].append(metrics.normalized_mutual_info_score(y, pred))
        scores['Homogeneity'].append(metrics.homogeneity_score(y, pred))

result_df = pd.DataFrame(scores)

#remake the DataFrame to have a single 'value' column for different scores
result_df1 = pd.melt(result_df, id_vars=['c', 'link'], value_vars=['rand_score', 'MI', 'AMI', 'NMI', 'Homogeneity'], var_name='score')

#plotting facet grid
g = sns.FacetGrid(result_df1, col='link', height=4, col_wrap=2, sharey=False)
g.map(sns.barplot, 'c', 'value', 'score', ci=None)
g.set_axis_labels('c', 'Score')
g.add_legend(title='Score')
plt.suptitle("Scores for Every Combination of Clusters and Linkage", y=1.02)
plt.show()



From the plot we see that the best parameters for agglomerative algorithm are n_clusters=5 and linkage='ward'. So, we make an agglomerative algorithm with these parameters to evaluate the cluster

In [ ]:
aggl=AgglomerativeClustering(n_clusters=5,linkage='ward')
pred=aggl.fit_predict(X,y)
df['pred']=pred
sns.histplot(data=df, x='pred', hue='oscar_winners')

We try to map the clusters predictions with the actual labels of the data

In [ ]:
pred = [1 if c!=2 else 0 for c in pred]
print(classification_report(y, pred))

From the classification report we see the poorness of the clustering algorithm

##DBSCAN

###Evaluate DBSCAN

In [ ]:
from sklearn.cluster import DBSCAN
from collections import defaultdict
from sklearn.metrics import adjusted_rand_score, mutual_info_score, adjusted_mutual_info_score, normalized_mutual_info_score, homogeneity_score


min_samples = 5
scores = defaultdict(list)
distances = ['cityblock', 'cosine', 'euclidean', 'l1', 'l2', 'manhattan']
eps = [5, 5.5, 6, 6.5, 7]

for e in eps:
    for d in distances:
        dbscan = DBSCAN(eps=e, min_samples=min_samples, metric=d)
        dbscan.fit(X)
        pred=dbscan.labels_
        df['pred'] = pred
        scores['e'].append(e)
        scores['metric'].append(d)
        scores['rand_score'].append(adjusted_rand_score(pred, y))
        scores['MI'].append(mutual_info_score(y, pred))
        scores['AMI'].append(adjusted_mutual_info_score(y, pred))
        scores['NMI'].append(normalized_mutual_info_score(y, pred))
        scores['Homogeneity'].append(homogeneity_score(y, pred))

result_df = pd.DataFrame(scores)

#remake the DataFrame to have a single 'value' column for different scores
result_df1 = pd.melt(result_df, id_vars=['e', 'metric'], value_vars=['rand_score', 'MI', 'AMI', 'NMI', 'Homogeneity'], var_name='score')

#Plotting facet grid
g = sns.FacetGrid(result_df1, col='metric', height=4, col_wrap=2, sharey=False)
g.map(sns.barplot, 'e', 'value', 'score', ci=None)
g.set_axis_labels('e', 'Score')
g.add_legend(title='Score')
plt.suptitle("Scores for Every Combination of eps and Metric", y=1.02)
plt.show()


We see that we have the best results for eps=7 and metric='euclidean'. So we make  a DBSCAN  with these parameters.

In [ ]:
dbscan=DBSCAN(eps=7, min_samples=5, metric='euclidean')
dbscan.fit(X)
pred=dbscan.labels_
df['pred']=pred
sns.histplot(data=df, x='pred', hue='oscar_winners')

From the plot we see the poorness of the clustering

In [ ]:
pred=[1 if (c==4 or c==6 or c==7) else 0 for c in pred ]
print(classification_report(y,pred))

Here the high accuracy may affect of the high presicion and recall for class 0.

So we do the clustering using the Kmean algorithm. The result is that moovies with oscar values have lower critics and moovies without oscar values have better critics.